<a href="https://colab.research.google.com/github/yusufbukarmaina/BeakerVolume/blob/main/CropMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Load the data from CSV
def load_data(file_path):
    """Load data from CSV file"""
    df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/cleaned_dataset.csv")
    return df

# Load the data
# Replace 'crop_data.csv' with your actual file path
file_path = 'cleaned_dataset.csv'
df = load_data(file_path)

# Display basic information
print("Dataset Info:")
print(df.info())
print("\nSample Data:")
print(df.head())

# Exploratory Data Analysis
print("\nSummary Statistics:")
print(df.describe())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Data Preprocessing
# Separate features and target
X = df.drop('Yield_tons_per_hectare', axis=1)
y = df['Yield_tons_per_hectare']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
boolean_cols = X.select_dtypes(include=['bool']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create preprocessing pipelines
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols),
        ('num', numerical_transformer, numerical_cols)
    ],
    remainder='passthrough'  # Pass through the boolean columns
)

# Split data into train and test sets (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create pipelines for each model including a Neural Network model
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])

# MLP/Neural Network Pipeline
# Using hidden_layer_sizes=(100, 50) for two hidden layers with 100 and 50 neurons respectively
mlp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(
        hidden_layer_sizes=(100, 50),
        activation='relu',
        solver='adam',
        alpha=0.0001,
        batch_size='auto',
        learning_rate='adaptive',
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42))
])

# Define models dictionary
models = {
    'Random Forest': rf_pipeline,
    'XGBoost': xgb_pipeline,
    'Gradient Boosting': gb_pipeline,
    'Neural Network (MLP)': mlp_pipeline
}

# 5-fold Cross Validation
cv_results = {}
print("\n5-Fold Cross Validation Results:")
for name, model in models.items():
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-cv_scores)
    cv_results[name] = rmse_scores
    print(f"{name} - Mean RMSE: {rmse_scores.mean():.4f}, Std Dev: {rmse_scores.std():.4f}")

# Train models on the full training set and evaluate on test set
results = {}
feature_importances = {}
print("\nModel Evaluation on Test Set:")

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Store results
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }

    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R²: {r2:.4f}")

    # Extract feature importances
    if hasattr(model.named_steps['model'], 'feature_importances_'):
        # Get feature names after preprocessing
        preprocessor = model.named_steps['preprocessor']

        # Get the feature names after one-hot encoding
        cat_feature_names = []
        if categorical_cols:
            cat_feature_names = preprocessor.transformers_[0][1].named_steps['onehot'].get_feature_names_out(categorical_cols).tolist()

        # Combine with numerical and boolean features
        feature_names = cat_feature_names + numerical_cols + boolean_cols

        # Get feature importances
        importances = model.named_steps['model'].feature_importances_

        # Create feature importance dataframe
        # Make sure the length of feature_names matches importances
        if len(feature_names) == len(importances):
            feature_importances[name] = pd.DataFrame({
                'Feature': feature_names,
                'Importance': importances
            }).sort_values('Importance', ascending=False)

            print("\n  Top 10 Feature Importances:")
            print(feature_importances[name].head(10))
        else:
            print(f"\n  Warning: Feature names length ({len(feature_names)}) doesn't match importances length ({len(importances)})")
            # Create a simple feature importance dataframe with indices
            feature_importances[name] = pd.DataFrame({
                'Feature': [f'Feature_{i}' for i in range(len(importances))],
                'Importance': importances
            }).sort_values('Importance', ascending=False)

            print("\n  Top 10 Feature Importances (using indices):")
            print(feature_importances[name].head(10))

    # For MLPRegressor, use permutation importance to get feature importance
    elif name == 'Neural Network (MLP)':
        # Apply preprocessor to get transformed feature matrix
        X_train_transformed = model.named_steps['preprocessor'].transform(X_train)

        # Get feature names after preprocessing
        preprocessor = model.named_steps['preprocessor']

        # Get the feature names after one-hot encoding
        cat_feature_names = []
        if categorical_cols:
            cat_feature_names = preprocessor.transformers_[0][1].named_steps['onehot'].get_feature_names_out(categorical_cols).tolist()

        # Combine with numerical and boolean features
        feature_names = cat_feature_names + numerical_cols + boolean_cols

        # Calculate permutation importance
        perm_importance = permutation_importance(
            model.named_steps['model'], X_train_transformed, y_train,
            n_repeats=10, random_state=42
        )

        # Get importance scores
        importances = perm_importance.importances_mean

        # Create feature importance dataframe
        # Make sure the length of feature_names matches importances
        if len(feature_names) == len(importances):
            feature_importances[name] = pd.DataFrame({
                'Feature': feature_names,
                'Importance': importances
            }).sort_values('Importance', ascending=False)

            print("\n  Top 10 Feature Importances (Permutation Method):")
            print(feature_importances[name].head(10))
        else:
            print(f"\n  Warning: Feature names length ({len(feature_names)}) doesn't match importances length ({len(importances)})")
            # Create a simple feature importance dataframe with indices
            feature_importances[name] = pd.DataFrame({
                'Feature': [f'Feature_{i}' for i in range(len(importances))],
                'Importance': importances
            }).sort_values('Importance', ascending=False)

            print("\n  Top 10 Feature Importances (Permutation Method, using indices):")
            print(feature_importances[name].head(10))

# Visualize results
plt.figure(figsize=(15, 10))

# Cross-validation scores
plt.subplot(2, 2, 1)
sns.boxplot(data=[cv_results[model] for model in models.keys()], width=0.5)
plt.xticks(range(len(models)), models.keys(), rotation=45)
plt.ylabel('RMSE Score')
plt.title('Cross-validation RMSE Scores')

# Test set metrics
plt.subplot(2, 2, 2)
metrics_df = pd.DataFrame(results).T
metrics_df = metrics_df.reset_index().melt(id_vars='index', var_name='Metric', value_name='Value')
sns.barplot(x='index', y='Value', hue='Metric', data=metrics_df)
plt.xlabel('Model')
plt.ylabel('Score')
plt.title('Test Set Metrics')
plt.legend(title='Metric')
plt.xticks(rotation=45)

# Feature importances - Focus on comparing Neural Network (MLP) with best model
best_traditional_model = min([m for m in results.keys() if m != 'Neural Network (MLP)'],
                            key=lambda x: results[x]['RMSE'])

plt.subplot(2, 2, 3)
if best_traditional_model in feature_importances:
    top_features = feature_importances[best_traditional_model].head(10)
    sns.barplot(x='Importance', y='Feature', data=top_features)
    plt.title(f'Top 10 Feature Importances for {best_traditional_model}')

plt.subplot(2, 2, 4)
if 'Neural Network (MLP)' in feature_importances:
    top_features = feature_importances['Neural Network (MLP)'].head(10)
    sns.barplot(x='Importance', y='Feature', data=top_features)
    plt.title('Top 10 Feature Importances for Neural Network (MLP)')

plt.tight_layout()
plt.savefig('model_comparison.png')
plt.close()

# Compare predicted vs actual values for the best model and Neural Network
best_model = min(results, key=lambda x: results[x]['RMSE'])
models_to_compare = ['Neural Network (MLP)', best_model] if best_model != 'Neural Network (MLP)' else ['Neural Network (MLP)', best_traditional_model]

plt.figure(figsize=(15, 6))
for i, model_name in enumerate(models_to_compare):
    plt.subplot(1, 2, i+1)
    model_pipeline = models[model_name]
    y_pred = model_pipeline.predict(X_test)

    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel('Actual Yield')
    plt.ylabel('Predicted Yield')
    plt.title(f'Actual vs Predicted Yield ({model_name})')

plt.tight_layout()
plt.savefig('actual_vs_predicted_comparison.png')
plt.close()

# Compare Neural Network with best traditional model
nn_model = models['Neural Network (MLP)']
best_trad_model = models[best_traditional_model]
nn_pred = nn_model.predict(X_test)
best_trad_pred = best_trad_model.predict(X_test)

# Create a comparison dataframe
comparison_df = pd.DataFrame({
    'Actual': y_test,
    'Neural Network Prediction': nn_pred,
    f'{best_traditional_model} Prediction': best_trad_pred,
    'NN Error': abs(y_test - nn_pred),
    f'{best_traditional_model} Error': abs(y_test - best_trad_pred)
})

# Calculate which model performs better for each instance
comparison_df['Better Model'] = comparison_df.apply(
    lambda row: 'Neural Network' if row['NN Error'] < row[f'{best_traditional_model} Error']
    else best_traditional_model if row['NN Error'] > row[f'{best_traditional_model} Error']
    else 'Equal', axis=1
)

# Count which model performs better
better_model_counts = comparison_df['Better Model'].value_counts()
print("\nModel Performance Comparison (instance-wise):")
print(better_model_counts)

# Plot distribution of errors
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(comparison_df['NN Error'], kde=True, color='blue', label='Neural Network')
sns.histplot(comparison_df[f'{best_traditional_model} Error'], kde=True, color='orange', label=best_traditional_model)
plt.xlabel('Absolute Error')
plt.ylabel('Frequency')
plt.title('Distribution of Prediction Errors')
plt.legend()

plt.subplot(1, 2, 2)
sns.boxplot(data=[comparison_df['NN Error'], comparison_df[f'{best_traditional_model} Error']])
plt.xticks([0, 1], ['Neural Network', best_traditional_model])
plt.ylabel('Absolute Error')
plt.title('Comparison of Error Distributions')

plt.tight_layout()
plt.savefig('error_comparison.png')
plt.close()

# Detailed statistical comparison
print("\nDetailed Statistical Comparison:")
for name in models_to_compare:
    model_pipeline = models[name]
    y_pred = model_pipeline.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Calculate additional metrics
    explained_variance = 1 - (np.var(y_test - y_pred) / np.var(y_test))
    max_error = np.max(np.abs(y_test - y_pred))
    median_absolute_error = np.median(np.abs(y_test - y_pred))

    print(f"\n{name} - Detailed Metrics:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R²: {r2:.4f}")
    print(f"  Explained Variance: {explained_variance:.4f}")
    print(f"  Maximum Error: {max_error:.4f}")
    print(f"  Median Absolute Error: {median_absolute_error:.4f}")

# Save the best model
from joblib import dump
best_overall_model = min(results, key=lambda x: results[x]['RMSE'])
dump(models[best_overall_model], 'best_crop_yield_model.joblib')
print(f"\nBest overall model ({best_overall_model}) saved as 'best_crop_yield_model.joblib'")

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Region                  10000 non-null  object 
 1   Soil_Type               10000 non-null  object 
 2   Crop                    10000 non-null  object 
 3   Rainfall_mm             10000 non-null  float64
 4   Temperature_Celsius     10000 non-null  float64
 5   Fertilizer_Used         10000 non-null  bool   
 6   Irrigation_Used         10000 non-null  bool   
 7   Weather_Condition       10000 non-null  object 
 8   Days_to_Harvest         10000 non-null  int64  
 9   Yield_tons_per_hectare  10000 non-null  float64
dtypes: bool(2), float64(3), int64(1), object(4)
memory usage: 644.7+ KB
None

Sample Data:
  Region Soil_Type     Crop  Rainfall_mm  Temperature_Celsius  \
0   west     peaty    wheat   507.158029            26.955093   
1  north      loam  